In [1]:
from pathlib import Path

ordner = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_2"

csv_dateien = sorted(ordner.rglob("*.csv"))

print("=== VORHANDENE CSV-DATEIEN ===")

for datei in csv_dateien:
    print("📄", datei.relative_to(ordner))

print(f"\nInsgesamt gefunden: {len(csv_dateien)} CSV-Dateien")

=== VORHANDENE CSV-DATEIEN ===
📄 Pipeline_Tag4/clean/bestellungen_april_clean.csv
📄 Pipeline_Tag4/raw/bestellungen_april_roh.csv
📄 bestellungen_april.csv
📄 bestellungen_april_roh.csv
📄 bestellungen_clean.csv
📄 bestellungen_pii.csv
📄 bestellungen_q1_roh.csv
📄 kunden.csv
📄 kunden_clean.csv
📄 produkte.csv
📄 produkte_clean.csv

Insgesamt gefunden: 11 CSV-Dateien


In [2]:
import pandas as pd

dateien = {
    "Kunden": "kunden_clean.csv",
    "Produkte": "produkte_clean.csv",
    "Bestellungen": "bestellungen_clean.csv"
}

daten = {}

for name, dateiname in dateien.items():
    df = pd.read_csv(ordner / dateiname)
    daten[name] = df

    print(f"\n=== {name.upper()} ===")
    print("Zeilen:", len(df))
    print("Spalten:", list(df.columns))
    print("Datentypen:")
    print(df.dtypes.to_string())
    print("-" * 50)


=== KUNDEN ===
Zeilen: 40
Spalten: ['kunde_id', 'name', 'ort', 'land']
Datentypen:
kunde_id    int64
name          str
ort           str
land          str
--------------------------------------------------

=== PRODUKTE ===
Zeilen: 12
Spalten: ['produkt_id', 'produktname', 'kategorie', 'preis_eur']
Datentypen:
produkt_id       int64
produktname        str
kategorie          str
preis_eur      float64
--------------------------------------------------

=== BESTELLUNGEN ===
Zeilen: 180
Spalten: ['bestell_id', 'datum', 'kunde_id', 'produkt_id', 'menge']
Datentypen:
bestell_id    int64
datum           str
kunde_id      int64
produkt_id    int64
menge         int64
--------------------------------------------------


In [3]:
# Primärschlüssel prüfen
schluessel = {
    "Kunden": "kunde_id",
    "Produkte": "produkt_id",
    "Bestellungen": "bestell_id"
}

for name, spalte in schluessel.items():
    df = daten[name]

    print(f"\n=== {name} ===")
    print("Doppelte IDs:", df[spalte].duplicated().sum())
    print("Fehlende IDs:", df[spalte].isna().sum())

# Fremdschlüssel prüfen
bestellungen = daten["Bestellungen"]
kunden = daten["Kunden"]
produkte = daten["Produkte"]

fehlende_kunden = ~bestellungen["kunde_id"].isin(kunden["kunde_id"])
fehlende_produkte = ~bestellungen["produkt_id"].isin(produkte["produkt_id"])

print("\n=== FREMDDSCHLÜSSEL ===")
print("Bestellungen ohne gültigen Kunden:", fehlende_kunden.sum())
print("Bestellungen ohne gültiges Produkt:", fehlende_produkte.sum())


=== Kunden ===
Doppelte IDs: 0
Fehlende IDs: 0

=== Produkte ===
Doppelte IDs: 0
Fehlende IDs: 0

=== Bestellungen ===
Doppelte IDs: 0
Fehlende IDs: 0

=== FREMDDSCHLÜSSEL ===
Bestellungen ohne gültigen Kunden: 0
Bestellungen ohne gültiges Produkt: 0


In [4]:
import sqlite3

# Datenbank im Ordner Woche_2 erstellen
db_pfad = ordner / "shop.db"

conn = sqlite3.connect(db_pfad)

# Fremdschlüsselprüfung aktivieren
conn.execute("PRAGMA foreign_keys = ON")

# Tabelle Kunden
conn.execute("""
CREATE TABLE IF NOT EXISTS kunden (
    kunde_id INTEGER PRIMARY KEY,
    name TEXT,
    ort TEXT,
    land TEXT
)
""")

# Tabelle Produkte
conn.execute("""
CREATE TABLE IF NOT EXISTS produkte (
    produkt_id INTEGER PRIMARY KEY,
    produktname TEXT,
    kategorie TEXT,
    preis_eur REAL
)
""")

# Tabelle Bestellungen
conn.execute("""
CREATE TABLE IF NOT EXISTS bestellungen (
    bestell_id INTEGER PRIMARY KEY,
    datum TEXT,
    kunde_id INTEGER,
    produkt_id INTEGER,
    menge INTEGER,
    FOREIGN KEY (kunde_id) REFERENCES kunden(kunde_id),
    FOREIGN KEY (produkt_id) REFERENCES produkte(produkt_id)
)
""")

conn.commit()

print("Datenbank erstellt:", db_pfad)

tabellen = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

print("\nVorhandene Tabellen:")
print(tabellen)

Datenbank erstellt: /Users/saschagolparvari/Desktop/Modul 5/Aufgaben/Woche_2/shop.db

Vorhandene Tabellen:
           name
0        kunden
1      produkte
2  bestellungen


In [5]:
# CSV-Daten in die SQLite-Tabellen übertragen

daten["Kunden"].to_sql(
    "kunden",
    conn,
    if_exists="append",
    index=False
)

daten["Produkte"].to_sql(
    "produkte",
    conn,
    if_exists="append",
    index=False
)

daten["Bestellungen"].to_sql(
    "bestellungen",
    conn,
    if_exists="append",
    index=False
)

# Zeilenzahlen kontrollieren
for tabelle in ["kunden", "produkte", "bestellungen"]:

    anzahl = conn.execute(
        f"SELECT COUNT(*) FROM {tabelle}"
    ).fetchone()[0]

    print(f"{tabelle}: {anzahl} Datensätze")

kunden: 40 Datensätze
produkte: 12 Datensätze
bestellungen: 180 Datensätze


In [6]:
# Primärschlüssel testen

# Eine bereits vorhandene Kunden-ID ermitteln
vorhandene_id = kunden["kunde_id"].iloc[0]

print("Teste doppelte Kunden-ID:", vorhandene_id)

try:
    conn.execute("""
        INSERT INTO kunden (kunde_id, name, ort, land)
        VALUES (?, ?, ?, ?)
    """, (
        int(vorhandene_id),
        "Testkunde",
        "Hagen",
        "DE"
    ))

    conn.commit()
    print("FEHLER: Doppelte ID wurde akzeptiert!")

except sqlite3.IntegrityError as e:
    print("TEST BESTANDEN!")
    print("SQLite verhindert doppelte Primärschlüssel.")
    print("Fehlermeldung:", e)

Teste doppelte Kunden-ID: 101
TEST BESTANDEN!
SQLite verhindert doppelte Primärschlüssel.
Fehlermeldung: UNIQUE constraint failed: kunden.kunde_id


In [7]:
# SQL-Abfragen: COUNT und WHERE

# 1. Gesamtzahl aller Bestellungen
sql_count = """
SELECT COUNT(*) AS anzahl_bestellungen
FROM bestellungen
"""

ergebnis_count = pd.read_sql_query(sql_count, conn)

print("=== COUNT: ALLE BESTELLUNGEN ===")
print(ergebnis_count)


# 2. Bestellungen mit einer Menge größer als 2
sql_where = """
SELECT *
FROM bestellungen
WHERE menge > 2
"""

ergebnis_where = pd.read_sql_query(sql_where, conn)

print("\n=== WHERE: BESTELLUNGEN MIT MENGE > 2 ===")
print(ergebnis_where)

print("\nGefundene Bestellungen:", len(ergebnis_where))

=== COUNT: ALLE BESTELLUNGEN ===
   anzahl_bestellungen
0                  180

=== WHERE: BESTELLUNGEN MIT MENGE > 2 ===
    bestell_id       datum  kunde_id  produkt_id  menge
0         5002  2026-04-01       123         502      3
1         5004  2026-04-01       120         504      4
2         5005  2026-04-01       101         504      3
3         5012  2026-04-02       135         508      4
4         5013  2026-04-02       101         511      4
..         ...         ...       ...         ...    ...
81        5173  2026-04-29       120         504      3
82        5175  2026-04-30       107         502      3
83        5177  2026-04-30       109         509      4
84        5179  2026-04-30       136         502      3
85        5180  2026-04-30       137         505      3

[86 rows x 5 columns]

Gefundene Bestellungen: 86


In [8]:
# SQL-JOIN: Umsatz pro Kunde berechnen

sql_join = """
SELECT
    k.kunde_id,
    k.name,
    k.ort,
    COUNT(b.bestell_id) AS anzahl_bestellungen,
    SUM(b.menge * p.preis_eur) AS gesamtumsatz

FROM bestellungen b

JOIN kunden k
    ON b.kunde_id = k.kunde_id

JOIN produkte p
    ON b.produkt_id = p.produkt_id

GROUP BY
    k.kunde_id,
    k.name,
    k.ort

ORDER BY gesamtumsatz DESC

LIMIT 5
"""

ergebnis_join = pd.read_sql_query(sql_join, conn)

print("=== TOP 5 KUNDEN NACH UMSATZ ===")
print(ergebnis_join.to_string(index=False))

=== TOP 5 KUNDEN NACH UMSATZ ===
 kunde_id         name        ort  anzahl_bestellungen  gesamtumsatz
      113 Mara Richter     Berlin                    8        2674.1
      103   Clara Kaya       Köln                    7        2671.9
      109   Ida Schulz Düsseldorf                    4        2306.6
      136   Lena Fuchs  Innsbruck                    9        2289.4
      115    Oona Wolf   Nürnberg                    6        2144.6


In [9]:
# pandas-Gegenprobe zur SQL-Umsatzberechnung

# Bestellungen mit Produkten verbinden
vergleich = bestellungen.merge(
    produkte,
    on="produkt_id",
    how="inner"
)

# Kundendaten hinzufügen
vergleich = vergleich.merge(
    kunden,
    on="kunde_id",
    how="inner"
)

# Umsatz je Bestellposition berechnen
vergleich["umsatz"] = (
    vergleich["menge"] * vergleich["preis_eur"]
)

# Umsatz und Bestellanzahl pro Kunde berechnen
pandas_ergebnis = (
    vergleich.groupby(["kunde_id", "name", "ort"])
    .agg(
        anzahl_bestellungen=("bestell_id", "count"),
        gesamtumsatz=("umsatz", "sum")
    )
    .reset_index()
    .sort_values("gesamtumsatz", ascending=False)
    .head(5)
)

print("=== PANDAS: TOP 5 KUNDEN ===")
print(pandas_ergebnis.to_string(index=False))

# SQL-Ergebnis mit pandas vergleichen
import numpy as np

vergleich_ok = (
    ergebnis_join["kunde_id"].tolist()
    == pandas_ergebnis["kunde_id"].tolist()
    and
    np.allclose(
        ergebnis_join["gesamtumsatz"],
        pandas_ergebnis["gesamtumsatz"]
    )
    and
    ergebnis_join["anzahl_bestellungen"].tolist()
    == pandas_ergebnis["anzahl_bestellungen"].tolist()
)

print("\n=== GEGENPROBE ===")
print("SQL und pandas stimmen überein:", vergleich_ok)

=== PANDAS: TOP 5 KUNDEN ===
 kunde_id         name        ort  anzahl_bestellungen  gesamtumsatz
      113 Mara Richter     Berlin                    8        2674.1
      103   Clara Kaya       Köln                    7        2671.9
      109   Ida Schulz Düsseldorf                    4        2306.6
      136   Lena Fuchs  Innsbruck                    9        2289.4
      115    Oona Wolf   Nürnberg                    6        2144.6

=== GEGENPROBE ===
SQL und pandas stimmen überein: True


Reflexion – SQLite-Datenbank shop.db

1. CSV-Dateien vs. SQLite-Datenbank

Ein Vorteil einer SQLite-Datenbank gegenüber einzelnen CSV-Dateien ist die bessere Kontrolle über die Datenintegrität.

Durch Primärschlüssel können doppelte IDs verhindert werden. Fremdschlüssel helfen dabei, sicherzustellen, dass Bestellungen gültigen Kunden und Produkten zugeordnet werden.

Außerdem können mehrere Tabellen mit SQL-Abfragen verbunden und ausgewertet werden. Gerade bei größeren Datenmengen ist das hilfreich.

2. Überprüfung der Ergebnisse

Ich habe drei bereinigte CSV-Dateien mit insgesamt 232 Datensätzen in eine SQLite-Datenbank übertragen.

Anschließend habe ich verschiedene SQL-Abfragen durchgeführt und die Umsätze der fünf umsatzstärksten Kunden berechnet.

Die Ergebnisse habe ich unabhängig mit pandas überprüft. Beide Berechnungen lieferten dieselben Ergebnisse.

3. Einsatz von KI

Für diese Aufgabe habe ich ChatGPT als Programmier- und Lernassistenten verwendet. GitHub Copilot kam nicht zum Einsatz.

ChatGPT hat mir die einzelnen Codeschritte vorgeschlagen und erklärt. Ich habe diese in Jupyter Notebook ausgeführt und die Ergebnisse kontrolliert.

Die KI hat mich besonders beim Erstellen der SQL-Tabellen, bei den JOIN-Abfragen und bei der pandas-Gegenprobe unterstützt.

4. Persönliche Erkenntnisse

Ich habe besser verstanden, wie Primärschlüssel und Fremdschlüssel funktionieren und warum sie wichtig sind.

Außerdem habe ich gelernt, dass SQL verschiedene Tabellen miteinander verbinden kann und dass sich die Ergebnisse mit pandas überprüfen lassen.

Die eigenständige Erstellung des Codes fällt mir noch schwer. Deshalb möchte ich weiterhin daran arbeiten, die einzelnen Befehle besser zu verstehen und zunehmend selbstständig anzuwenden.